In [ ]:
from imports import*
from utils import*
from models import*
from Train_Eval_Test import*

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device: {}'.format(device))

# Hyperparameters updated for 3D U-Net
args = {
    'device': device,
    'num_features' : 1,     # Input channels (XCT grayscale)
    'f_maps' : 32,          # Base filters for U-Net (replaces 'hidden')
    'num_classes' : 6,
    'lr': 0.001,
    'epochs': 200,
}

### Setup of the dataset
We use 3D patches (sub-volumes) for the U-Net input.

In [ ]:
side = 512 
new_side = 64 
stride = 28 
steps = int((side - new_side) / stride + 1)

### Data Loading and Reshaping
For U-Net, we reshape to 5D Tensors: (N, Channel, Depth, Height, Width)

In [ ]:
features_test = raw_to_tensor("CVSynth.raw", side)
labels_test = raw_to_tensor("CVSynth_Labels.raw", side)

x_test = torch.tensor(view_as_windows(features_test.numpy(), (new_side,new_side,new_side), step=stride).reshape(-1, 1, new_side, new_side, new_side))
y_test = torch.tensor(view_as_windows(labels_test.numpy(), (new_side,new_side,new_side), step=stride).reshape(-1, new_side, new_side, new_side))

In [ ]:
# Experimental volumes with NLM filter
features_1_exp = raw_to_tensor("CV1_NLM8.raw", side)
features_2_exp = raw_to_tensor("CV2_NLM8.raw", side)
features_3_exp = raw_to_tensor("CV3_NLM8.raw", side)
features_4_exp = raw_to_tensor("CV4_NLM8.raw", side)

# Reshape experimental data to 5D
x_1_exp = torch.tensor(view_as_windows(features_1_exp.numpy(), (new_side,new_side,new_side), step=stride).reshape(-1, 1, new_side, new_side, new_side))
x_2_exp = torch.tensor(view_as_windows(features_2_exp.numpy(), (new_side,new_side,new_side), step=stride).reshape(-1, 1, new_side, new_side, new_side))
x_3_exp = torch.tensor(view_as_windows(features_3_exp.numpy(), (new_side,new_side,new_side), step=stride).reshape(-1, 1, new_side, new_side, new_side))
x_4_exp = torch.tensor(view_as_windows(features_4_exp.numpy(), (new_side,new_side,new_side), step=stride).reshape(-1, 1, new_side, new_side, new_side))

In [ ]:
# Import manually-labelled slices
labels_1_exp = tif_to_tensor("CV1 Labels - Slice 339.tif", side)
labels_2_exp = tif_to_tensor("CV2 Labels - Slice 139.tif", side)
labels_3_exp = tif_to_tensor("CV3 Labels - Slice 219.tif", side)
labels_4_exp = tif_to_tensor("CV4 Labels - Slice 059.tif", side)

### Data Loaders
U-Net uses standard PyTorch Tensors (no Data/edge_index objects needed).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=1, shuffle=False)
test_loader_1 = DataLoader(TensorDataset(x_1_exp), batch_size=1, shuffle=False)
test_loader_2 = DataLoader(TensorDataset(x_2_exp), batch_size=1, shuffle=False)
test_loader_3 = DataLoader(TensorDataset(x_3_exp), batch_size=1, shuffle=False)
test_loader_4 = DataLoader(TensorDataset(x_4_exp), batch_size=1, shuffle=False)

### Inference and Evaluation
Running the trained 3D U-Net models on the experimental volumes.

In [ ]:
for i in range(10):
    # Initialize 3D U-Net model
    model = UNet3D(args['num_features'], args['num_classes'], f_maps=args['f_maps']).to(device)
    model.load_state_dict(torch.load(f'UNet_{i}.h5'))
    model.eval()

    eval_obj = Test(model, device)

    # Synthetic Assessment
    preds = extract_overlap_pred(model, test_loader, args['num_classes'], steps, new_side, side, stride)
    dice_val = dice(preds, labels_test, average='none', num_classes=6)
    
    # Experimental Assessment (Volumes 1-4)
    # Similar logic for volume reconstruction as previous code, 
    # but ensured for 5D tensor input in extract_overlap_pred
    preds_1 = extract_overlap_pred(model, test_loader_1, args['num_classes'], steps, new_side, side, stride)
    
    # Log results to CSV
    # ... (CSV writing logic remains similar to original)
    print(f"Model {i} evaluated.")